# 10.9 Model Explainability

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 10: Model Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-10-model-optimization/10.09_model-explainability.ipynb)

Companion code for Section 10.9 of the book. Run the cells in order from top to bottom.

**Example 10.9.1** · bash

```bash
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings('ignore')

data = load_breast_cancer()
X, y = data.data, data.target
feature_names = list(data.feature_names)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                     max_depth=4, random_state=42)
model.fit(X_tr, y_tr)
print(f'Test accuracy: {model.score(X_te, y_te):.4f}')

# ── 1. Permutation Feature Importance ─────────────────────────────
perm_result = permutation_importance(
    model, X_te, y_te, n_repeats=30, random_state=42, n_jobs=-1
)
perm_imp = pd.DataFrame({
    'feature':    feature_names,
    'importance': perm_result.importances_mean,
    'std':        perm_result.importances_std
}).sort_values('importance', ascending=False)

print('\nTop 10 Permutation Importances:')
print(perm_imp.head(10).round(4).to_string(index=False))

# ── 2. Partial Dependence Plots ───────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Model Explainability - Global Methods', fontsize=14, fontweight='bold')

# Feature importance bar chart
ax = axes[0]
perm_imp.head(10).sort_values('importance').plot(
    kind='barh', x='feature', y='importance', xerr='std',
    ax=ax, color='#1976D2', edgecolor='white', capsize=3, legend=False)
ax.set_title('Permutation Feature Importance\n(top 10)'); ax.grid(alpha=0.3, axis='x')

# PDP for top 2 features
top2 = perm_imp.head(2)['feature'].tolist()
top2_idx = [feature_names.index(f) for f in top2]

for ax, feat_idx in zip(axes[1:3], top2_idx):
    feat_name = feature_names[feat_idx]
    display = PartialDependenceDisplay.from_estimator(
        model, X_te, features=[feat_idx], feature_names=feature_names,
        target=1, ax=ax
    )
    ax.set_title(f'Partial Dependence:\n{feat_name}')
    ax.grid(alpha=0.3)

# 2D PDP interaction
ax = axes[3]
display_2d = PartialDependenceDisplay.from_estimator(
    model, X_te, features=[(top2_idx[0], top2_idx[1])],
    feature_names=feature_names, target=1, ax=ax
)
ax.set_title(f'2D PDP Interaction:\n{top2[0]} × {top2[1]}')

plt.tight_layout()
plt.savefig('explainability_global.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 3. Individual Prediction Analysis ─────────────────────────────
# Show which features drove specific predictions
sample_idx = [0, 5, 10]  # Three specific test samples
print('\nLocal analysis: feature contributions for specific predictions')
for idx in sample_idx:
    x_sample = X_te[idx]
    pred_prob = model.predict_proba([x_sample])[0]
    pred_class= model.predict([x_sample])[0]
    true_label= y_te[idx]
    print(f'\n  Sample {idx}: True={data.target_names[true_label]}  Pred={data.target_names[pred_class]}
P(benign)={pred_prob[1]:.4f}')
    # Show top deviating features from training mean
    deviations = np.abs(x_sample - X_tr.mean(axis=0)) / (X_tr.std(axis=0)+1e-8)
    top_dev_idx = np.argsort(deviations)[::-1][:4]
    for fi in top_dev_idx:
        print(f'    {feature_names[fi]:35s}: {x_sample[fi]:>10.4f}  (train mean: {X_tr[:,fi].mean():.4f}
z={deviations[fi]:+.2f})')
```